# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Note: Not subscripting, treat as object

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate the record sets, show their `@id`s and the fields within them using their `@id` as well as their data types and descriptions.

In [ ]:
# List all record sets by their `@id`, then show their fields
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"Record Set Name: {rs.name} | @id: {rs.id}")
        print("  Fields:")
        for f in rs.fields:
            print(f"    Field: {f.name} | @id: {f.id} | DataType: {f.data_type} | Description: {f.description}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we will auto-discover all record sets and load them into dataframes using their @id
# We'll print the columns and show the head of the main (first) record set.

dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    # Each record_set can be loaded using its @id
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Example rows:")
    display(df.head(2))
    print()

# Select the main record set for further analysis (usually the biggest one)
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# We'll demonstrate EDA using a numeric field ('phq9_total_score' commonly measures depression severity)
# and a grouping categorical field (e.g. 'gender'), referencing only by their field @ids.
# You may have to adjust these IDs based on your dataset's overview.

import numpy as np

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Try to find a numeric field for analysis
    # (For this dataset, 'phq9_total_score' appears frequently as a total PHQ-9 depression score)
    # We'll search columns including 'phq9' or 'PHQ' or default to the first float/int column.

    possible_numeric_fields = [col for col in df.columns if 'phq' in col.lower() or 'score' in col.lower()]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
    else:
        numeric_field = df.select_dtypes(include=[np.number]).columns[0]

    print(f"Using numeric field: {numeric_field}")
    
    # Select a threshold for demonstration (e.g., 10 is often a cutoff for moderate depression)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, normalized_col]].head())

    # Find a grouping field (e.g., 'gender' or 'GAD7_severity')
    group_field = None
    for col in df.columns:
        if 'gender' in col.lower() or 'sex' in col.lower():
            group_field = col
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count'])
        print(f"\nGrouped data by {group_field}:")
        display(grouped_df)
    else:
        print("No grouping field like 'gender' or 'sex' found for aggregation.")
else:
    print("No valid record set found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll include basic matplotlib and seaborn plots as examples.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group field, if available
    if group_field is not None:
        plt.figure(figsize=(7,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print("No main record set found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and perform basic analysis on the [A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library, referencing all entities by their Croissant `@id`. We reviewed record set structures, filtered and normalized key assessment scores, and visualized key distributions.

This workflow can be adapted to other FAIR datasets described by Croissant schemas by simply changing the Croissant schema URL.